# `helpers/llm/client.py` — Playground

The **shared LLM standard** every gate (2–5) reuses. Two functions:

| Function | Notes |
|---|---|
| `build_agent(output_type, system_prompt, *, model=, temperature=, retries=)` | Construct a configured pydantic-ai `Agent` **once** per gate. Cheap Haiku default; `model=` overridable. |
| `run_agent(agent, user_prompt)` | Run one call → validated output model, or `None` on any failure (fetcher contract). Loop-aware (works here in Jupyter). |

**Pattern:** define a pydantic output model → `build_agent(...)` at module load → `run_agent(...)` per candidate.

Cells below: (1) zero-cost wiring via `TestModel`, (2) a real live call, (3) per-gate model override, (4) failure → `None`, (5) free-play. Live calls need `ANTHROPIC_API_KEY` in `.env`.

In [1]:
import sys
import pathlib

# Add 02_intelligence/ so `helpers.llm.client` imports like the gates do.
# Walk upward to find it, so this works whether the notebook is launched
# from the repo root or from its own folder.
intelligence_dir = next(
    p / 'backend' / '02_intelligence'
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / 'backend' / '02_intelligence').is_dir()
)
if str(intelligence_dir) not in sys.path:
    sys.path.insert(0, str(intelligence_dir))

from pydantic import BaseModel, Field
from helpers.llm.client import build_agent, run_agent, DEFAULT_MODEL, DEFAULT_TEMPERATURE

print(f'default model: {DEFAULT_MODEL}  |  default temperature: {DEFAULT_TEMPERATURE}')

default model: anthropic:claude-haiku-4-5-20251001  |  default temperature: 0.0


---
## 1. Wiring check with `TestModel` — zero API cost

`TestModel` is pydantic-ai's fake model: it fabricates a valid instance of your output type **without calling the API**. Use `agent.override(model=...)` to swap it in. This proves `build_agent` + `run_agent` are wired correctly before spending a cent.

In [2]:
from pydantic_ai.models.test import TestModel

class Demo(BaseModel):
    ok: bool = Field(description='Did it work?')
    note: str = Field(description='A short note')

agent = build_agent(Demo, 'You are a wiring test.')
with agent.override(model=TestModel()):
    result = run_agent(agent, 'ping')

print(type(result).__name__, '→', result)

Demo → ok=True note='Wiring test successful - ping received and responded to'


---
## 2. A real live call

Same agent, no override — this hits Claude (Haiku by default). The model fills your pydantic fields and `run_agent` returns the validated object. Needs `ANTHROPIC_API_KEY`.

In [3]:
class Sentiment(BaseModel):
    label: str = Field(description="One of: 'positive', 'negative', 'neutral'")
    confidence: float = Field(description='0.0 to 1.0')

sentiment_agent = build_agent(
    Sentiment,
    'You classify the sentiment of a single financial headline.',
)

run_agent(sentiment_agent, 'Headline: Company beats earnings, raises full-year guidance.')

Sentiment(label='positive', confidence=0.95)

---
## 3. Per-gate model override

The whole point of the shared standard: each gate can trade cost for quality by passing `model=`. Gate 2 might stay on cheap Haiku; a harder gate could pass a stronger model here. Same `build_agent` / `run_agent` either way.

In [4]:
# Build the SAME agent against a different model + warmer temperature.
# (Swap the model string for whatever you want to try.)
strong_agent = build_agent(
    Sentiment,
    'You classify the sentiment of a single financial headline.',
    model='anthropic:claude-haiku-4-5-20251001',  # ← change me, e.g. a Sonnet/Opus id
    temperature=0.3,
)

run_agent(strong_agent, 'Headline: Regulators open a preliminary inquiry; company says it is cooperating.')

Sentiment(label='negative', confidence=0.75)

---
## 4. Failure → `None` (the fetcher contract)

`run_agent` never raises. On any failure (auth, network, validation) it prints a `[llm]` warning and returns `None` — the caller decides what `None` means for its gate. Here we force a bad model id to see it.

In [5]:
broken_agent = build_agent(Demo, 'test', model='anthropic:not-a-real-model')
result = run_agent(broken_agent, 'ping')
print('returned:', result)  # expect: None, after a [llm] warning

[llm] call failed — status_code: 404, model_name: not-a-real-model, body: {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: not-a-real-model'}, 'request_id': 'req_011CbZyzgJMbMvJUXa6oKTe7'}
returned: None


---
## Free-play

Define your own output model + system prompt and try it. (Use `TestModel` via `agent.override(...)` if you want to iterate without API cost.)